In [3]:
import os
import sys

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def analyze_runs(directory="."):
    # Если передана папка в аргументах:
    # python3 analyze.py build
    if len(sys.argv) > 1 and not sys.argv[1].startswith("-"):
        directory = sys.argv[1]

    N_PARTICLES = 1_000_000
    BEAM_ENERGY_MEV = 5
    TOTAL_BEAM_ENERGY = N_PARTICLES * BEAM_ENERGY_MEV

    thicknesses = [19]
    results = []

    print(f"Ищу файлы в папке: {os.path.abspath(directory)}")
    files = [f for f in os.listdir(directory) if f.endswith(".root") and "Output_Run_" in f]
    files.sort(key=lambda x: int(x.split("_")[-1].split(".")[0]))

    if not files:
        print(f"Ошибка: файлы .root не найдены в '{directory}'")
        print("Попробуй запустить так: python3 analyze.py build")
        return

    plots_dir = os.path.join(directory, "plots")
    os.makedirs(plots_dir, exist_ok=True)

    # Общий график для всех кривых
    plt.figure(figsize=(10, 6))

    for i, file_name in enumerate(files):
        path = os.path.join(directory, file_name)

        try:
            current_thick = thicknesses[i] if i < len(thicknesses) else thicknesses[-1]

            root_file = uproot.open(path)
            hist = root_file["pdd"]

            values, edges = hist.to_numpy()
            centers = 0.5 * (edges[:-1] + edges[1:])
            bin_width = edges[1] - edges[0]

            # Берем только область продукта по толщине
            limit_bin = int(current_thick / bin_width)
            limit_bin = min(limit_bin, len(values))

            product_data = values[:limit_bin]
            product_depth = centers[:limit_bin]
            useful_data = product_data[product_data > 0]

            if len(useful_data) == 0:
                print(f"Предупреждение: в файле {file_name} нет ненулевых данных в пределах толщины {current_thick} мм")
                continue

            v_max = np.percentile(useful_data, 95)
            v_min = np.percentile(useful_data, 5)

            dur = v_max / v_min if v_min > 0 else float("inf")
            efficiency = (np.sum(product_data) / TOTAL_BEAM_ENERGY) * 100.0
            max_dose = np.max(product_data)

            results.append({
                "File": file_name,
                "Thick(mm)": current_thick,
                "DUR (95/5%)": round(dur, 3),
                "Efficiency (%)": round(efficiency, 4),
                "Max Dose": round(float(max_dose), 4),
            })

            # Добавляем на общий график
            plt.plot(product_depth, product_data, linewidth=1.5, label=f"{current_thick} мм")

            # Отдельный график для каждого файла
            plt.figure(figsize=(9, 5.5))
            plt.plot(product_depth, product_data, linewidth=2)
            plt.xlabel("Глубина, мм")
            plt.ylabel("Поглощённая доза, отн. ед.")
            plt.title(f"Глубинное распределение дозы\n{file_name} | толщина слоя = {current_thick} мм")
            plt.grid(True, alpha=0.3)
            plt.tight_layout()

            single_plot_name = os.path.join(
                plots_dir,
                f"pdd_{current_thick:02d}mm.png"
            )
            plt.savefig(single_plot_name, dpi=300)
            plt.close()

        except Exception as e:
            print(f"Ошибка в файле {file_name}: {e}")

    # Сохраняем общий график
    if results:
        plt.xlabel("Глубина, мм")
        plt.ylabel("Поглощённая доза, отн. ед.")
        plt.title("Глубинные распределения поглощённой дозы")
        plt.grid(True, alpha=0.3)
        plt.legend(title="Толщина слоя", fontsize=8)
        plt.tight_layout()

        combined_plot_name = os.path.join(plots_dir, "pdd_all_runs.png")
        plt.savefig(combined_plot_name, dpi=300)
        plt.close()

        print(f"\n[ГОТОВО] Общий график сохранён: {combined_plot_name}")
        print(f"[ГОТОВО] Индивидуальные графики сохранены в папку: {plots_dir}")
    else:
        print("Нет данных для построения графиков.")
        return

    # Таблица результатов
    df = pd.DataFrame(results)

    print("\n" + "=" * 80)
    print(df.to_string(index=False))
    print("=" * 80)

    # output_name = os.path.join(directory, "simulation_results.csv")
    # df.to_csv(output_name, index=False)
    # print(f"\n[ГОТОВО] Таблица сохранена в файл: {output_name}")


if __name__ == "__main__":
    analyze_runs("build")

Ищу файлы в папке: /Users/pavelsimko/Documents/cub-lab/build

[ГОТОВО] Общий график сохранён: build/plots/pdd_all_runs.png
[ГОТОВО] Индивидуальные графики сохранены в папку: build/plots

             File  Thick(mm)  DUR (95/5%)  Efficiency (%)   Max Dose
Output_Run_0.root         19        1.522        112.1646 350892.516
